# When Should a Language Model Trust Itself?
## Colab notebook for evaluation, tables, and figures

This notebook reproduces the evaluation pipeline used in the paper on same-model self-verification as a confidence signal.

It is organized in three stages:
1. configure the experiment and run evaluation;
2. merge per-run outputs into summary metrics;
3. generate paper tables and figures from the saved results.


## 1) Setup

In [5]:
# Install dependencies
# Colab already includes PyTorch, so we install the libraries used by the notebook.
!pip -q install -U numpy scipy pandas scikit-learn datasets sentencepiece matplotlib

In [6]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate transformers

## 2) Mount Drive

In [1]:
# Optional: mount Google Drive if you want outputs to persist across Colab runtime restarts.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab. Skipping Drive mount.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3) Imports + GPU check

In [2]:
import os, json, math, time, glob, gc
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 4) Experiment config

In [4]:
# ============================
# Config (edit these)
# ============================

# Use a local Colab directory by default. Change this to a Drive path if you want persistent storage.
OUTPUT_DIR = "/content/self_verify_runs"
SEED = 42

MODELS = [
    "microsoft/phi-2",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-7B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
]

PROMPT_VARIANTS = [
    "default",
    "audit_v1",
]

RUN_DATASETS = ["truthfulqa_mc", "ai2_arc_challenge"]

SAVE_BATCH_SIZE = 100
GPU_PAIRS_BATCH = 8
GPU_SELF_BATCH  = 8
MC_MAX_TOKENS   = 256
SELF_MAX_TOKENS = 256

DTYPE = "float16"  # "float16" | "bfloat16" | "float32"

ENABLE_4BIT_FOR_LARGE_MODELS = True
LARGE_MODEL_SUBSTRINGS = ["7B", "8B", "13B", "14B", "32B", "70B"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODELS:", MODELS)
print("PROMPT_VARIANTS:", PROMPT_VARIANTS)

OUTPUT_DIR: /content/self_verify_runs
MODELS: ['microsoft/phi-2', 'Qwen/Qwen2.5-1.5B-Instruct', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'Qwen/Qwen2.5-7B-Instruct', 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B']
PROMPT_VARIANTS: ['default', 'audit_v1']


## 5) Dataset configs (STRICT loading)

In [5]:
# ============================
# Dataset configuration (STRICT)
# ============================

@dataclass(frozen=True)
class DatasetCfg:
    key: str
    dataset_id: str
    config: Optional[str]
    split: str
    revision: Optional[str] = None
    strict_config: bool = True

DATASETS = {
    "truthfulqa_mc": DatasetCfg(
        key="truthfulqa_mc",
        dataset_id="EleutherAI/truthful_qa_mc",
        config=None,
        split="validation",
        revision="refs/convert/parquet",
        strict_config=False
    ),
    "ai2_arc_challenge": DatasetCfg(
        key="ai2_arc_challenge",
        dataset_id="allenai/ai2_arc",
        config="ARC-Challenge",
        split="test",
        revision=None,              # DO NOT set parquet revision here
        strict_config=True
    ),
}

def load_dataset_strict(cfg: DatasetCfg):
    # Fail fast if config isn't available
    if cfg.strict_config and cfg.config is not None:
        configs = get_dataset_config_names(cfg.dataset_id)
        print(f"[CHECK] Available configs for {cfg.dataset_id}: {configs}")
        assert cfg.config in configs, (
            f"Requested config={cfg.config!r} not in {configs}. "
            "Refusing to silently fall back."
        )

    kwargs = {"split": cfg.split}
    if cfg.revision is not None:
        kwargs["revision"] = cfg.revision

    if cfg.config is None:
        ds = load_dataset(cfg.dataset_id, **kwargs)
        loaded_config = "(none)"
    else:
        ds = load_dataset(cfg.dataset_id, cfg.config, **kwargs)
        loaded_config = cfg.config

    print(f"[INFO] Loaded {cfg.dataset_id} | config={loaded_config} | split={cfg.split} | revision={cfg.revision or '(none)'} | rows={len(ds)}")
    return ds


## 6) Dataset adapters + shuffle order

In [6]:
# ============================
# Dataset adapters
# ============================

def _normalize_arc_answer_key(answer_key: Any, labels: List[str]) -> Optional[str]:
    if answer_key is None:
        return None
    ak = str(answer_key).strip()
    if ak in labels:
        return ak
    if ak.isdigit():
        k = int(ak)
        if 1 <= k <= 26:
            cand = chr(ord("A") + k - 1)
            if cand in labels:
                return cand
    return None

def adapt_row(dataset_key: str, row: Dict[str, Any], qid: int) -> Dict[str, Any]:
    if dataset_key == "truthfulqa_mc":
        question = row["question"]
        choices = list(row["choices"])
        gold_index = int(row["label"])
        return {"qid": qid, "question": question, "choices": choices, "gold_index": gold_index}

    if dataset_key == "ai2_arc_challenge":
        # common schema: row["question"] is dict(stem, choices=[{label,text},...])
        if isinstance(row.get("question", None), dict) and "stem" in row["question"]:
            question = row["question"]["stem"]
            choice_objs = row["question"]["choices"]
            choices = [c["text"] for c in choice_objs]
            labels  = [c["label"] for c in choice_objs]
            ak_norm = _normalize_arc_answer_key(row.get("answerKey", None), labels)
            gold_index = labels.index(ak_norm) if ak_norm is not None else None
            return {"qid": qid, "question": question, "choices": choices, "gold_index": gold_index}

        # fallback schema
        question = row["question"] if isinstance(row.get("question", None), str) else str(row.get("question"))
        ch = row["choices"]
        choices = list(ch["text"])
        labels  = list(ch["label"])
        ak_norm = _normalize_arc_answer_key(row.get("answerKey", None), labels)
        gold_index = labels.index(ak_norm) if ak_norm is not None else None
        return {"qid": qid, "question": question, "choices": choices, "gold_index": gold_index}

    raise ValueError(f"Unknown dataset_key: {dataset_key}")

def filter_valid_rows(dataset_key: str, ds):
    keep = []
    for i in range(len(ds)):
        it = adapt_row(dataset_key, ds[i], qid=i)
        gi = it["gold_index"]
        if gi is None:
            continue
        if 0 <= gi < len(it["choices"]):
            keep.append(i)
    kept = ds.select(keep)
    print(f"[INFO] filter_valid_rows({dataset_key}): kept {len(kept)}/{len(ds)}")
    return kept

def load_or_create_order(path: str, n: int, seed: int) -> np.ndarray:
    if os.path.exists(path):
        order = np.load(path)
        assert len(order) == n, f"Existing order len={len(order)} but dataset n={n}"
        return order
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    np.save(path, order)
    return order

def prepare_dataset(cfg: DatasetCfg, seed: int) -> Any:
    ds = load_dataset_strict(cfg)
    ds = filter_valid_rows(cfg.key, ds)
    order_p = os.path.join(OUTPUT_DIR, f"order__{cfg.key}__N{len(ds)}__seed{seed}.npy")
    order = load_or_create_order(order_p, len(ds), seed)
    ds = ds.select(order.tolist())
    print(f"[INFO] Prepared {cfg.key}: rows(after shuffle)={len(ds)}")
    return ds


## 7) Prompts

In [7]:

# ============================
# Prompts
# ============================

def prompt_mc(question: str, choices: List[str]) -> str:
    lines = ["Question:", question.strip(), "", "Choices:"]
    for i, ch in enumerate(choices):
        lines.append(f"{i}. {ch.strip()}")
    lines.append("")
    lines.append("Answer:")
    return "\n".join(lines)

def prompt_selfeval(question: str, proposed_answer: str, variant: str = "default") -> str:
    # Note: ends with "Answer: " (trailing space) to stabilize tokenization.
    q = question.strip()
    a = proposed_answer.strip()

    if variant == "default":
        return (
            "You are evaluating whether a proposed answer to a multiple-choice question is correct.\n"
            f"Question: {q}\n"
            f"Proposed answer: {a}\n\n"
            "Is the proposed answer correct? Respond with exactly one token: True or False.\n"
            "Answer: "
        )

    if variant == "audit_v1":
        return (
            "You are an answer auditor. Determine whether the proposed answer is actually supported by the question.\n"
            f"Question: {q}\n"
            f"Proposed answer: {a}\n\n"
            "Is the proposed answer correct? Respond with exactly one token: True or False.\n"
            "Answer: "
        )

    raise ValueError(f"Unknown self-eval prompt variant: {variant}")


## 8) Metrics

In [8]:
# ============================
# Metrics
# ============================

def softmax_np(x: Sequence[float]) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def brier(conf: np.ndarray, y: np.ndarray) -> float:
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return float(np.mean((conf - y) ** 2))

def ece(conf: np.ndarray, y: np.ndarray, n_bins: int = 10) -> float:
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    e = 0.0
    n = len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            mask = (conf >= lo) & (conf < hi)
        else:
            mask = (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        acc_bin = y[mask].mean()
        conf_bin = conf[mask].mean()
        e += (mask.sum() / n) * abs(acc_bin - conf_bin)
    return float(e)

def auroc_score(scores: np.ndarray, labels: np.ndarray) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)
    from sklearn.metrics import roc_auc_score
    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(roc_auc_score(labels, scores))

def aurc(conf: np.ndarray, y: np.ndarray) -> float:
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    idx = np.argsort(-conf)
    y_s = y[idx]
    n = len(y_s)
    coverages = np.arange(1, n + 1) / n
    risks = 1.0 - (np.cumsum(y_s) / np.arange(1, n + 1))
    cov = np.concatenate([[0.0], coverages])
    rsk = np.concatenate([[1.0], risks])
    return float(np.trapezoid(rsk, cov))


## 9) Model loading + robust True/False token IDs

In [9]:
# ============================
# Model loading + robust True/False IDs
# ============================

tokenizer = None
model = None
MODEL_DEVICE = None
SELF_TRUE_IDS = []
SELF_FALSE_IDS = []

def _single_token_id(s: str) -> Optional[int]:
    ids = tokenizer(s, add_special_tokens=False).input_ids
    return ids[0] if len(ids) == 1 else None

def init_self_eval_token_ids():
    global SELF_TRUE_IDS, SELF_FALSE_IDS
    true_variants  = ["True", " True", "TRUE", " TRUE"]
    false_variants = ["False", " False", "FALSE", " FALSE"]

    true_ids, false_ids = [], []
    for s in true_variants:
        tid = _single_token_id(s)
        if tid is not None:
            true_ids.append(tid)
    for s in false_variants:
        fid = _single_token_id(s)
        if fid is not None:
            false_ids.append(fid)

    # fallback (rare): use 1/0
    if not true_ids or not false_ids:
        true_ids, false_ids = [], []
        for s in ["1", " 1"]:
            tid = _single_token_id(s)
            if tid is not None:
                true_ids.append(tid)
        for s in ["0", " 0"]:
            fid = _single_token_id(s)
            if fid is not None:
                false_ids.append(fid)

    if not true_ids or not false_ids:
        raise RuntimeError("No usable True/False token IDs found — check tokenizer/prompt formatting.")

    SELF_TRUE_IDS  = list(dict.fromkeys(true_ids))
    SELF_FALSE_IDS = list(dict.fromkeys(false_ids))

def is_large_model(model_id: str) -> bool:
    model_id_upper = model_id.upper()
    return any(tag in model_id_upper for tag in LARGE_MODEL_SUBSTRINGS)

def load_model_and_tokenizer(model_id: str):
    global tokenizer, model, MODEL_DEVICE
    print("\n" + "="*100)
    print("[LOAD]", model_id)

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    # Ensure tokenizer has pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load config explicitly and patch pad_token_id if missing
    config = AutoConfig.from_pretrained(model_id)
    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        config.pad_token_id = tokenizer.pad_token_id

    torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[DTYPE]

    quantization_config = None
    use_4bit = bool(torch.cuda.is_available() and ENABLE_4BIT_FOR_LARGE_MODELS and is_large_model(model_id))
    if use_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch_dtype if torch_dtype in [torch.float16, torch.bfloat16] else torch.float16,
        )
        print("[INFO] Using 4-bit quantization")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        torch_dtype=torch_dtype if torch.cuda.is_available() and quantization_config is None else None,
        quantization_config=quantization_config,
        device_map="auto" if torch.cuda.is_available() else None,
    ).eval()

    MODEL_DEVICE = next(model.parameters()).device
    init_self_eval_token_ids()

    print("[INFO] DEVICE:", MODEL_DEVICE)
    print("[INFO] pad_token_id:", tokenizer.pad_token_id, "| eos_token_id:", tokenizer.eos_token_id)
    print("[INFO] config.pad_token_id:", getattr(config, "pad_token_id", None))
    print("[INFO] SELF_TRUE_IDS:", SELF_TRUE_IDS)
    print("[INFO] SELF_FALSE_IDS:", SELF_FALSE_IDS)

def unload_model():
    global tokenizer, model, MODEL_DEVICE
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    model = None
    tokenizer = None
    MODEL_DEVICE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 10) (Optional) Diagnostic: next-token preferences

In [10]:
# ============================
# Diagnostic: inspect next-token preferences for the self-eval prompt
# (Run this once per model if you're paranoid.)
# ============================

def top_next_tokens(prompt: str, k: int = 20):
    enc = tokenizer([prompt], return_tensors="pt").to(MODEL_DEVICE)
    out = model(**enc)
    last = enc["attention_mask"].sum(1).item() - 1
    logits = out.logits[0, last, :]
    top = torch.topk(logits, k=k)
    toks = tokenizer.convert_ids_to_tokens(top.indices.tolist())
    return list(zip(toks, top.values.detach().cpu().tolist()))

# Example:
# load_model_and_tokenizer("microsoft/phi-2")
# p = prompt_selfeval("2+2=?", "4")
# print(top_next_tokens(p, 30)[:10])


## 11) Scoring functions

In [11]:
# ============================
# Scoring functions
# ============================

@torch.no_grad()
def log_likelihood_batch(pairs: List[Tuple[str, str]]) -> List[float]:
    # SUM log-likelihood of continuation tokens given the prompt.
    prompts = [p for p, _ in pairs]
    conts   = [c for _, c in pairs]

    enc_p = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                      max_length=MC_MAX_TOKENS, add_special_tokens=False)
    enc_c = tokenizer(conts, return_tensors="pt", padding=True, truncation=True,
                      max_length=MC_MAX_TOKENS, add_special_tokens=False)

    input_ids_list, attn_list, cont_lens = [], [], []
    for i in range(len(pairs)):
        p_ids = enc_p["input_ids"][i]
        c_ids = enc_c["input_ids"][i]
        p_len = int((p_ids != tokenizer.pad_token_id).sum().item())
        c_len = int((c_ids != tokenizer.pad_token_id).sum().item())
        p_ids, c_ids = p_ids[:p_len], c_ids[:c_len]
        cont_lens.append(c_len)

        full = torch.cat([p_ids, c_ids], dim=0)
        if full.numel() > MC_MAX_TOKENS:
            full = full[-MC_MAX_TOKENS:]

        input_ids_list.append(full)
        attn_list.append(torch.ones_like(full))

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id).to(MODEL_DEVICE)
    attn      = torch.nn.utils.rnn.pad_sequence(attn_list, batch_first=True, padding_value=0).to(MODEL_DEVICE)

    out = model(input_ids=input_ids, attention_mask=attn)
    logp = torch.log_softmax(out.logits, dim=-1)

    scores = []
    for i in range(len(pairs)):
        c_len = cont_lens[i]
        if c_len == 0:
            scores.append(float("-inf"))
            continue
        seq_len = int(attn[i].sum().item())
        cont_ids = input_ids[i, seq_len - c_len: seq_len]
        start = seq_len - c_len

        s = 0.0
        for j, tok in enumerate(cont_ids):
            pos = start + j
            if pos == 0:
                continue
            s += float(logp[i, pos - 1, tok].item())
        scores.append(s)

    return scores

@torch.no_grad()
def p_true_batch(self_prompts: List[str]) -> np.ndarray:
    # Robust P(True) using logsumexp over token variants, then sigmoid(logit_true - logit_false)
    true_ids  = torch.tensor(SELF_TRUE_IDS,  device=MODEL_DEVICE, dtype=torch.long)
    false_ids = torch.tensor(SELF_FALSE_IDS, device=MODEL_DEVICE, dtype=torch.long)

    probs = []
    for i in range(0, len(self_prompts), GPU_SELF_BATCH):
        chunk = self_prompts[i:i+GPU_SELF_BATCH]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=SELF_MAX_TOKENS)
        input_ids = enc["input_ids"].to(MODEL_DEVICE)
        attn      = enc["attention_mask"].to(MODEL_DEVICE)

        out = model(input_ids=input_ids, attention_mask=attn)
        last_pos = attn.sum(dim=1) - 1
        next_logits = out.logits[torch.arange(out.logits.size(0), device=MODEL_DEVICE), last_pos, :]

        lt = torch.logsumexp(next_logits.index_select(1, true_ids), dim=1)
        lf = torch.logsumexp(next_logits.index_select(1, false_ids), dim=1)
        p = torch.sigmoid(lt - lf)

        probs.extend(p.detach().float().cpu().numpy().tolist())

    return np.asarray(probs, dtype=np.float32)


## 12) Evaluation per slice

In [12]:
# ============================
# Evaluation (per slice)
# ============================

def evaluate_slice(dataset_key: str, ds, start: int, end: int, prompt_variant: str = "default") -> pd.DataFrame:
    items = [adapt_row(dataset_key, ds[i], qid=i) for i in range(start, end)]

    # build (prompt, choice) pairs
    pairs = []
    pair_map = []
    for li, it in enumerate(items):
        pmc = prompt_mc(it["question"], it["choices"])
        for ci, choice in enumerate(it["choices"]):
            pairs.append((pmc, choice))
            pair_map.append((li, ci))

    # score all choices
    scores_all = [None] * len(pairs)
    for j in range(0, len(pairs), GPU_PAIRS_BATCH):
        chunk = pairs[j:j+GPU_PAIRS_BATCH]
        s = log_likelihood_batch(chunk)
        for k, val in enumerate(s):
            scores_all[j+k] = float(val)

    # reshape to per-question list
    choice_scores = [[0.0]*len(items[i]["choices"]) for i in range(len(items))]
    for idx, (li, ci) in enumerate(pair_map):
        choice_scores[li][ci] = float(scores_all[idx])

    preds_sum, preds_avg = [], []
    cprob_sum, cprob_avg = [], []
    margin_sum, margin_avg = [], []
    probs_sum_debug, probs_avg_debug, tok_len_debug = [], [], []

    for li, it in enumerate(items):
        scores = np.asarray(choice_scores[li], dtype=np.float64)

        probs_sum = softmax_np(scores.tolist())
        pred_s = int(probs_sum.argmax())
        sp = np.sort(probs_sum)
        marg_s = float(sp[-1] - sp[-2]) if len(sp) >= 2 else float("nan")

        tok_lens = np.asarray([len(tokenizer(ch, add_special_tokens=False).input_ids) for ch in it["choices"]], dtype=np.float64)
        scores_avg = scores / np.maximum(tok_lens, 1.0)
        probs_avg = softmax_np(scores_avg.tolist())
        pred_a = int(probs_avg.argmax())
        ap = np.sort(probs_avg)
        marg_a = float(ap[-1] - ap[-2]) if len(ap) >= 2 else float("nan")

        preds_sum.append(pred_s); cprob_sum.append(float(probs_sum[pred_s])); margin_sum.append(marg_s)
        preds_avg.append(pred_a); cprob_avg.append(float(probs_avg[pred_a])); margin_avg.append(marg_a)

        probs_sum_debug.append(json.dumps([float(p) for p in probs_sum.tolist()]))
        probs_avg_debug.append(json.dumps([float(p) for p in probs_avg.tolist()]))
        tok_len_debug.append(json.dumps([float(t) for t in tok_lens.tolist()]))

    # self-verification uses AVG prediction (primary)
    self_prompts = [
        prompt_selfeval(items[li]["question"], items[li]["choices"][preds_avg[li]], variant=prompt_variant)
        for li in range(len(items))
    ]
    c_verbs = p_true_batch(self_prompts)

    rows = []
    for li, it in enumerate(items):
        y_sum = int(preds_sum[li] == it["gold_index"])
        y_avg = int(preds_avg[li] == it["gold_index"])
        rows.append({
            "qid": int(it["qid"]),
            "prompt_variant": prompt_variant,
            "y_avg": y_avg,
            "y_sum": y_sum,
            "pred_index_avg": int(preds_avg[li]),
            "pred_index_sum": int(preds_sum[li]),
            "gold_index": int(it["gold_index"]),
            "c_prob_avg": float(cprob_avg[li]),
            "c_prob_sum": float(cprob_sum[li]),
            "margin_avg": float(margin_avg[li]),
            "margin_sum": float(margin_sum[li]),
            "c_verb": float(c_verbs[li]),
            "choice_scores": json.dumps([float(s) for s in choice_scores[li]]),
            "choice_probs_avg": probs_avg_debug[li],
            "choice_probs_sum": probs_sum_debug[li],
            "choice_tok_lens": tok_len_debug[li],
        })

    return pd.DataFrame(rows)


## 13) Resumable batch runner + merge + metrics

In [13]:
# ============================
# Batch runner (resumable) + merge + metrics
# ============================

def safe_name(s: str) -> str:
    return s.replace("/", "__").replace(":", "_").replace(" ", "_")

def batch_path(results_dir: str, b: int) -> str:
    return os.path.join(results_dir, f"batch_{b:03d}.csv")

def run_all_batches(dataset_key: str, ds, results_dir: str, prompt_variant: str = "default"):
    os.makedirs(results_dir, exist_ok=True)
    n = len(ds)
    nb = math.ceil(n / SAVE_BATCH_SIZE)
    print(f"[INFO] Total questions={n} | SAVE_BATCH_SIZE={SAVE_BATCH_SIZE} | num_batches={nb} | prompt_variant={prompt_variant}")

    for b in range(nb):
        out_csv = batch_path(results_dir, b)
        start = b * SAVE_BATCH_SIZE
        end   = min((b + 1) * SAVE_BATCH_SIZE, n)
        exp_rows = end - start

        if os.path.exists(out_csv):
            try:
                df_existing = pd.read_csv(out_csv)
                if len(df_existing) == exp_rows:
                    print(f"[SKIP] batch {b:03d} rows {start}:{end} (exists+complete)")
                    continue
                print(f"[WARN] batch {b:03d} incomplete ({len(df_existing)}/{exp_rows}) → overwrite")
            except Exception:
                print(f"[WARN] batch {b:03d} unreadable → overwrite")

        print(f"[RUN ] batch {b:03d} rows {start}:{end}")
        t0 = time.time()
        df_batch = evaluate_slice(dataset_key, ds, start, end, prompt_variant=prompt_variant)
        df_batch.to_csv(out_csv, index=False)
        print(f"[SAVE] {out_csv} | rows={len(df_batch)} | {(time.time()-t0)/60:.2f} min")

def merge_batches(results_dir: str, out_csv: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(results_dir, "batch_*.csv")))
    assert files, f"No batch files in {results_dir}"
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True).sort_values("qid").reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    print(f"[INFO] Merged {len(files)} batches → {out_csv} | rows={len(df)}")
    return df

def compute_metrics(df: pd.DataFrame) -> Dict[str, Any]:
    y_avg = df["y_avg"].to_numpy(np.int32)
    y_sum = df["y_sum"].to_numpy(np.int32)
    c_prob_avg = df["c_prob_avg"].to_numpy(np.float64)
    c_prob_sum = df["c_prob_sum"].to_numpy(np.float64)
    c_verb = df["c_verb"].to_numpy(np.float64)

    m = {
        "n": int(len(df)),
        "prompt_variant": str(df["prompt_variant"].iloc[0]) if "prompt_variant" in df.columns and len(df) else "default",
        "accuracy_avg": float(y_avg.mean()),
        "accuracy_sum": float(y_sum.mean()),
        "corr_cprob_avg_cverb": float(np.corrcoef(c_prob_avg, c_verb)[0, 1]) if len(df) > 1 else float("nan"),
        "corr_cprob_sum_cverb": float(np.corrcoef(c_prob_sum, c_verb)[0, 1]) if len(df) > 1 else float("nan"),
        "brier_c_prob_avg": brier(c_prob_avg, y_avg.astype(np.float64)),
        "brier_c_prob_sum": brier(c_prob_sum, y_sum.astype(np.float64)),
        "brier_c_verb": brier(c_verb, y_avg.astype(np.float64)),
        "ece10_c_prob_avg": ece(c_prob_avg, y_avg.astype(np.float64), 10),
        "ece10_c_prob_sum": ece(c_prob_sum, y_sum.astype(np.float64), 10),
        "ece10_c_verb": ece(c_verb, y_avg.astype(np.float64), 10),
        "auroc_c_prob_avg": auroc_score(c_prob_avg, y_avg),
        "auroc_c_verb": auroc_score(c_verb, y_avg),
        "auroc_c_prob_sum": auroc_score(c_prob_sum, y_sum),
        "aurc_c_prob_avg": aurc(c_prob_avg, y_avg.astype(np.float64)),
        "aurc_c_verb": aurc(c_verb, y_avg.astype(np.float64)),
        "aurc_c_prob_sum": aurc(c_prob_sum, y_sum.astype(np.float64)),
    }
    return m


## 14) Smoke test (run this first)

In [14]:
import importlib.util

def has_bitsandbytes():
    return importlib.util.find_spec("bitsandbytes") is not None

print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes installed:", has_bitsandbytes())

CUDA available: True
bitsandbytes installed: True


In [15]:
# ============================
# SMOKE TEST (recommended)
# Run 50 items for 1 model + 1 dataset to validate everything works.
# ============================

SMOKE_MODEL = MODELS[-1]          # use the larger model for validation
SMOKE_DATASET = RUN_DATASETS[0]
SMOKE_PROMPT_VARIANT = PROMPT_VARIANTS[0]

cfg = DATASETS[SMOKE_DATASET]
ds = prepare_dataset(cfg, SEED)

# reduce to 50 for smoke
ds_smoke = ds.select(range(min(50, len(ds))))
print("[SMOKE] dataset rows:", len(ds_smoke), "| prompt_variant:", SMOKE_PROMPT_VARIANT)

load_model_and_tokenizer(SMOKE_MODEL)
df_smoke = evaluate_slice(cfg.key, ds_smoke, 0, len(ds_smoke), prompt_variant=SMOKE_PROMPT_VARIANT)
print(df_smoke.head(3))

m_smoke = compute_metrics(df_smoke)
print("[SMOKE metrics]\n", json.dumps(m_smoke, indent=2))

unload_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0000.parquet:   0%|          | 0.00/99.5k [00:00<?, ?B/s]

Generating validation split: 0 examples [00:00, ? examples/s]

[INFO] Loaded EleutherAI/truthful_qa_mc | config=(none) | split=validation | revision=refs/convert/parquet | rows=684
[INFO] filter_valid_rows(truthfulqa_mc): kept 684/684
[INFO] Prepared truthfulqa_mc: rows(after shuffle)=684
[SMOKE] dataset rows: 50 | prompt_variant: default

[LOAD] deepseek-ai/DeepSeek-R1-Distill-Llama-8B


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[INFO] DEVICE: cuda:0
[INFO] pad_token_id: 128001 | eos_token_id: 128001
[INFO] config.pad_token_id: 128001
[INFO] SELF_TRUE_IDS: [2575, 21260]
[INFO] SELF_FALSE_IDS: [4139, 31451]
   qid prompt_variant  y_avg  y_sum  pred_index_avg  pred_index_sum  \
0    0        default      0      0               2               2   
1    1        default      0      0               3               3   
2    2        default      0      0               0               0   

   gold_index  c_prob_avg  c_prob_sum  margin_avg  margin_sum    c_verb  \
0           0    0.966167    1.000000    0.932465     1.00000  0.742188   
1           0    0.330873    0.463728    0.003857     0.07932  0.652344   
2           1    0.320571    0.333333    0.000000     0.00000  0.960938   

                                       choice_scores  \
0  [-75.8125, -114.7972412109375, -25.90954589843...   
1  [-39.75535202026367, -38.82663536071777, -70.5...   
2                 [-38.625, -38.625, -75.0, -38.625]   

        

## 15) Full run (5 models × 2 datasets)

In [16]:
# ============================
# FULL RUN — TruthfulQA-MC (all models × prompt variants)
# ============================

summary_truthful = []

dkey = "truthfulqa_mc"
cfg = DATASETS[dkey]

print("\n" + "#"*110)
print("[DATASET]", dkey)
ds = prepare_dataset(cfg, SEED)

ds_dir = os.path.join(OUTPUT_DIR, dkey)
os.makedirs(ds_dir, exist_ok=True)

for model_id in MODELS:
    for prompt_variant in PROMPT_VARIANTS:
        mdir = os.path.join(ds_dir, safe_name(model_id), f"prompt_variant={safe_name(prompt_variant)}")
        batches_dir = os.path.join(mdir, "batches")
        os.makedirs(batches_dir, exist_ok=True)

        final_csv = os.path.join(mdir, "results.csv")
        metrics_path = os.path.join(mdir, "metrics.json")
        config_path  = os.path.join(mdir, "config.json")

        load_model_and_tokenizer(model_id)

        with open(config_path, "w") as f:
            json.dump({
                "dataset_key": cfg.key,
                "dataset_id": cfg.dataset_id,
                "dataset_config": cfg.config,
                "dataset_split": cfg.split,
                "dataset_revision": cfg.revision,
                "n": int(len(ds)),
                "seed": SEED,
                "SAVE_BATCH_SIZE": SAVE_BATCH_SIZE,
                "GPU_PAIRS_BATCH": GPU_PAIRS_BATCH,
                "GPU_SELF_BATCH": GPU_SELF_BATCH,
                "MC_MAX_TOKENS": MC_MAX_TOKENS,
                "SELF_MAX_TOKENS": SELF_MAX_TOKENS,
                "DTYPE": DTYPE,
                "model_id": model_id,
                "prompt_variant": prompt_variant,
                "ENABLE_4BIT_FOR_LARGE_MODELS": ENABLE_4BIT_FOR_LARGE_MODELS,
                "SELF_TRUE_IDS": SELF_TRUE_IDS,
                "SELF_FALSE_IDS": SELF_FALSE_IDS,
            }, f, indent=2)

        run_all_batches(cfg.key, ds, batches_dir, prompt_variant=prompt_variant)
        df_final = merge_batches(batches_dir, final_csv)

        m = compute_metrics(df_final)
        m["model_id"] = model_id
        m["dataset_key"] = cfg.key
        m["prompt_variant"] = prompt_variant

        with open(metrics_path, "w") as f:
            json.dump(m, f, indent=2)

        print("[DONE]", cfg.key, "|", model_id, "|", prompt_variant)
        print(json.dumps(m, indent=2))

        summary_truthful.append(m)
        unload_model()

df_truthful = pd.DataFrame(summary_truthful).sort_values(
    ["model_id", "prompt_variant"], ascending=[True, True]
).reset_index(drop=True)
truthful_csv = os.path.join(OUTPUT_DIR, "summary_metrics__truthfulqa_mc.csv")
df_truthful.to_csv(truthful_csv, index=False)

print("\n" + "="*110)
print("[SAVED]", truthful_csv)
df_truthful


##############################################################################################################
[DATASET] truthfulqa_mc
[INFO] Loaded EleutherAI/truthful_qa_mc | config=(none) | split=validation | revision=refs/convert/parquet | rows=684
[INFO] filter_valid_rows(truthfulqa_mc): kept 684/684
[INFO] Prepared truthfulqa_mc: rows(after shuffle)=684

[LOAD] microsoft/phi-2


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] DEVICE: cuda:0
[INFO] pad_token_id: 128001 | eos_token_id: 128001
[INFO] config.pad_token_id: 128001
[INFO] SELF_TRUE_IDS: [2575, 21260]
[INFO] SELF_FALSE_IDS: [4139, 31451]
[INFO] Total questions=684 | SAVE_BATCH_SIZE=100 | num_batches=7 | prompt_variant=default
[RUN ] batch 000 rows 0:100
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_000.csv | rows=100 | 0.10 min
[RUN ] batch 001 rows 100:200
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_001.csv | rows=100 | 0.10 min
[RUN ] batch 002 rows 200:300
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_002.csv | rows=100 | 0.10 min
[RUN ] batch 003 rows 300:400
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_003.csv | rows=10

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] DEVICE: cuda:0
[INFO] pad_token_id: 128001 | eos_token_id: 128001
[INFO] config.pad_token_id: 128001
[INFO] SELF_TRUE_IDS: [2575, 21260]
[INFO] SELF_FALSE_IDS: [4139, 31451]
[INFO] Total questions=684 | SAVE_BATCH_SIZE=100 | num_batches=7 | prompt_variant=audit_v1
[RUN ] batch 000 rows 0:100
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_000.csv | rows=100 | 0.10 min
[RUN ] batch 001 rows 100:200
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_001.csv | rows=100 | 0.10 min
[RUN ] batch 002 rows 200:300
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_002.csv | rows=100 | 0.10 min
[RUN ] batch 003 rows 300:400
[SAVE] /content/self_verify_runs/truthfulqa_mc/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_003.csv | ro

,n,prompt_variant,accuracy_avg,accuracy_sum,corr_cprob_avg_cverb,corr_cprob_sum_cverb,brier_c_prob_avg,brier_c_prob_sum,brier_c_verb,ece10_c_prob_avg,ece10_c_prob_sum,ece10_c_verb,auroc_c_prob_avg,auroc_c_verb,auroc_c_prob_sum,aurc_c_prob_avg,aurc_c_verb,aurc_c_prob_sum,model_id,dataset_key
0,684,audit_v1,0.438596,0.377193,0.200484,0.230422,0.243812,0.356311,0.316046,0.074695,0.351472,0.267264,0.611024,0.620486,0.633639,0.488159,0.500229,0.508237,Qwen/Qwen2.5-1.5B-Instruct,truthfulqa_mc
1,684,default,0.438596,0.377193,0.166345,0.184503,0.243812,0.356311,0.355189,0.074695,0.351472,0.306594,0.611024,0.547687,0.633639,0.488159,0.542757,0.508237,Qwen/Qwen2.5-1.5B-Instruct,truthfulqa_mc
2,684,audit_v1,0.529240,0.555556,0.107781,0.156765,0.260162,0.271172,0.371794,0.123007,0.251810,0.360576,0.593314,0.664836,0.742452,0.416213,0.367213,0.266373,Qwen/Qwen2.5-7B-Instruct,truthfulqa_mc
3,684,default,0.529240,0.555556,0.160962,0.148884,0.260162,0.271172,0.363455,0.123007,0.251810,0.351327,0.593314,0.667346,0.742452,0.416213,0.370833,0.266373,Qwen/Qwen2.5-7B-Instruct,truthfulqa_mc
4,684,audit_v1,0.264620,0.209064,-0.028070,0.164277,0.194323,0.344199,0.487818,0.039885,0.391857,0.532424,0.562174,0.372066,0.503497,0.690516,0.799598,0.781508,TinyLlama/TinyLlama-1.1B-Chat-v1.0,truthfulqa_mc
5,684,default,0.264620,0.209064,-0.098466,0.131303,0.194323,0.344199,0.551889,0.039885,0.391857,0.590622,0.562174,0.362691,0.503497,0.690516,0.807318,0.781508,TinyLlama/TinyLlama-1.1B-Chat-v1.0,truthfulqa_mc
6,684,audit_v1,0.400585,0.377193,0.006586,-0.011846,0.419188,0.549626,0.390051,0.398963,0.556267,0.347664,0.614407,0.524430,0.577929,0.513852,0.597451,0.581350,deepseek-ai/DeepSeek-R1-Distill-Llama-8B,truthfulqa_mc
7,684,default,0.400585,0.377193,-0.010204,-0.006318,0.419188,0.549626,0.391803,0.398963,0.556267,0.340269,0.614407,0.494014,0.577929,0.513852,0.604693,0.581350,deepseek-ai/DeepSeek-R1-Distill-Llama-8B,truthfulqa_mc
8,684,audit_v1,0.467836,0.444444,0.246087,0.168929,0.255459,0.288623,0.241395,0.108794,0.236570,0.067307,0.590007,0.623957,0.658977,0.478318,0.460786,0.410109,microsoft/phi-2,truthfulqa_mc
9,684,default,0.467836,0.444444,0.250115,0.159904,0.255459,0.288623,0.242031,0.108794,0.236570,0.073444,0.590007,0.608036,0.658977,0.478318,0.465381,0.410109,microsoft/phi-2,truthfulqa_mc


In [17]:

# ============================
# FULL RUN — ARC-Challenge (all models × prompt variants)
# ============================

summary_arc = []

dkey = "ai2_arc_challenge"
cfg = DATASETS[dkey]

print("\n" + "#"*110)
print("[DATASET]", dkey)
ds = prepare_dataset(cfg, SEED)

ds_dir = os.path.join(OUTPUT_DIR, dkey)
os.makedirs(ds_dir, exist_ok=True)

for model_id in MODELS:
    for prompt_variant in PROMPT_VARIANTS:
        mdir = os.path.join(ds_dir, safe_name(model_id), f"prompt_variant={safe_name(prompt_variant)}")
        batches_dir = os.path.join(mdir, "batches")
        os.makedirs(batches_dir, exist_ok=True)

        final_csv = os.path.join(mdir, "results.csv")
        metrics_path = os.path.join(mdir, "metrics.json")
        config_path  = os.path.join(mdir, "config.json")

        load_model_and_tokenizer(model_id)

        with open(config_path, "w") as f:
            json.dump({
                "dataset_key": cfg.key,
                "dataset_id": cfg.dataset_id,
                "dataset_config": cfg.config,
                "dataset_split": cfg.split,
                "dataset_revision": cfg.revision,
                "n": int(len(ds)),
                "seed": SEED,
                "SAVE_BATCH_SIZE": SAVE_BATCH_SIZE,
                "GPU_PAIRS_BATCH": GPU_PAIRS_BATCH,
                "GPU_SELF_BATCH": GPU_SELF_BATCH,
                "MC_MAX_TOKENS": MC_MAX_TOKENS,
                "SELF_MAX_TOKENS": SELF_MAX_TOKENS,
                "DTYPE": DTYPE,
                "model_id": model_id,
                "prompt_variant": prompt_variant,
                "ENABLE_4BIT_FOR_LARGE_MODELS": ENABLE_4BIT_FOR_LARGE_MODELS,
                "SELF_TRUE_IDS": SELF_TRUE_IDS,
                "SELF_FALSE_IDS": SELF_FALSE_IDS,
            }, f, indent=2)

        run_all_batches(cfg.key, ds, batches_dir, prompt_variant=prompt_variant)
        df_final = merge_batches(batches_dir, final_csv)

        m = compute_metrics(df_final)
        m["model_id"] = model_id
        m["dataset_key"] = cfg.key
        m["prompt_variant"] = prompt_variant

        with open(metrics_path, "w") as f:
            json.dump(m, f, indent=2)

        print("[DONE]", cfg.key, "|", model_id, "|", prompt_variant)
        print(json.dumps(m, indent=2))

        summary_arc.append(m)
        unload_model()

df_arc = pd.DataFrame(summary_arc).sort_values(
    ["model_id", "prompt_variant"], ascending=[True, True]
).reset_index(drop=True)
arc_csv = os.path.join(OUTPUT_DIR, "summary_metrics__ai2_arc_challenge.csv")
df_arc.to_csv(arc_csv, index=False)

print("\n" + "="*110)
print("[SAVED]", arc_csv)
df_arc


##############################################################################################################
[DATASET] ai2_arc_challenge


README.md: 0.00B [00:00, ?B/s]

[CHECK] Available configs for allenai/ai2_arc: ['ARC-Challenge', 'ARC-Easy']


ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] DEVICE: cuda:0
[INFO] pad_token_id: 128001 | eos_token_id: 128001
[INFO] config.pad_token_id: 128001
[INFO] SELF_TRUE_IDS: [2575, 21260]
[INFO] SELF_FALSE_IDS: [4139, 31451]
[INFO] Total questions=1172 | SAVE_BATCH_SIZE=100 | num_batches=12 | prompt_variant=default
[RUN ] batch 000 rows 0:100
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_000.csv | rows=100 | 0.09 min
[RUN ] batch 001 rows 100:200
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_001.csv | rows=100 | 0.09 min
[RUN ] batch 002 rows 200:300
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch_002.csv | rows=100 | 0.10 min
[RUN ] batch 003 rows 300:400
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=default/batches/batch

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO] DEVICE: cuda:0
[INFO] pad_token_id: 128001 | eos_token_id: 128001
[INFO] config.pad_token_id: 128001
[INFO] SELF_TRUE_IDS: [2575, 21260]
[INFO] SELF_FALSE_IDS: [4139, 31451]
[INFO] Total questions=1172 | SAVE_BATCH_SIZE=100 | num_batches=12 | prompt_variant=audit_v1
[RUN ] batch 000 rows 0:100
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_000.csv | rows=100 | 0.09 min
[RUN ] batch 001 rows 100:200
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_001.csv | rows=100 | 0.09 min
[RUN ] batch 002 rows 200:300
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/batch_002.csv | rows=100 | 0.10 min
[RUN ] batch 003 rows 300:400
[SAVE] /content/self_verify_runs/ai2_arc_challenge/deepseek-ai__DeepSeek-R1-Distill-Llama-8B/prompt_variant=audit_v1/batches/

,n,prompt_variant,accuracy_avg,accuracy_sum,corr_cprob_avg_cverb,corr_cprob_sum_cverb,brier_c_prob_avg,brier_c_prob_sum,brier_c_verb,ece10_c_prob_avg,ece10_c_prob_sum,ece10_c_verb,auroc_c_prob_avg,auroc_c_verb,auroc_c_prob_sum,aurc_c_prob_avg,aurc_c_verb,aurc_c_prob_sum,model_id,dataset_key
0,1172,audit_v1,0.492321,0.572526,0.022493,0.195868,0.271754,0.268380,0.288640,0.118615,0.215839,0.275248,0.557386,0.770169,0.693358,0.471588,0.295729,0.273030,Qwen/Qwen2.5-1.5B-Instruct,ai2_arc_challenge
1,1172,default,0.492321,0.572526,0.018378,0.197633,0.271754,0.268380,0.287209,0.118615,0.215839,0.271747,0.557386,0.764798,0.693358,0.471588,0.298322,0.273030,Qwen/Qwen2.5-1.5B-Instruct,ai2_arc_challenge
2,1172,audit_v1,0.601536,0.694539,0.055021,0.222098,0.267706,0.203848,0.252671,0.166062,0.154837,0.252422,0.555047,0.878698,0.752690,0.363891,0.148573,0.154460,Qwen/Qwen2.5-7B-Instruct,ai2_arc_challenge
3,1172,default,0.601536,0.694539,0.063411,0.239719,0.267706,0.203848,0.238271,0.166062,0.154837,0.238678,0.555047,0.886431,0.752690,0.363891,0.143323,0.154460,Qwen/Qwen2.5-7B-Instruct,ai2_arc_challenge
4,1172,audit_v1,0.279010,0.272184,0.031468,-0.058976,0.221287,0.333545,0.453183,0.080313,0.335716,0.502089,0.483919,0.525084,0.536741,0.731402,0.702420,0.713921,TinyLlama/TinyLlama-1.1B-Chat-v1.0,ai2_arc_challenge
5,1172,default,0.279010,0.272184,-0.004608,-0.060919,0.221287,0.333545,0.491571,0.080313,0.335716,0.537951,0.483919,0.524648,0.536741,0.731402,0.700172,0.713921,TinyLlama/TinyLlama-1.1B-Chat-v1.0,ai2_arc_challenge
6,1172,audit_v1,0.278157,0.255973,-0.091220,-0.057194,0.426823,0.581665,0.359874,0.383394,0.575789,0.320031,0.510809,0.489043,0.532517,0.721500,0.737874,0.723792,deepseek-ai/DeepSeek-R1-Distill-Llama-8B,ai2_arc_challenge
7,1172,default,0.278157,0.255973,-0.038265,-0.049586,0.426823,0.581665,0.370029,0.383394,0.575789,0.339122,0.510809,0.463121,0.532517,0.721500,0.751202,0.723792,deepseek-ai/DeepSeek-R1-Distill-Llama-8B,ai2_arc_challenge
8,1172,audit_v1,0.534130,0.628840,0.124620,0.162422,0.268708,0.227587,0.209902,0.133886,0.131144,0.079839,0.547444,0.754605,0.693239,0.427488,0.278032,0.233367,microsoft/phi-2,ai2_arc_challenge
9,1172,default,0.534130,0.628840,0.193089,0.192660,0.268708,0.227587,0.204241,0.133886,0.131144,0.064484,0.547444,0.758847,0.693239,0.427488,0.274538,0.233367,microsoft/phi-2,ai2_arc_challenge


In [ ]:
# ============================
# MERGE RESULTS — TruthfulQA + ARC
# ============================

summary_files = [
    os.path.join(OUTPUT_DIR, "summary_metrics__truthfulqa_mc.csv"),
    os.path.join(OUTPUT_DIR, "summary_metrics__ai2_arc_challenge.csv"),
]

dfs = []
for f in summary_files:
    if os.path.exists(f):
        print("[LOAD]", f)
        dfs.append(pd.read_csv(f))
    else:
        print("[MISSING]", f)

df_summary = pd.concat(dfs, ignore_index=True)

# nicer ordering
sort_cols = ["dataset_key", "model_id", "prompt_variant"] if "prompt_variant" in df_summary.columns else ["dataset_key", "accuracy_avg"]
asc = [True, True, True] if "prompt_variant" in df_summary.columns else [True, False]
df_summary = df_summary.sort_values(sort_cols, ascending=asc).reset_index(drop=True)

summary_csv = os.path.join(OUTPUT_DIR, "summary_metrics.csv")
df_summary.to_csv(summary_csv, index=False)

print("\n" + "="*110)
print("[FINAL SUMMARY SAVED]", summary_csv)

df_summary

In [ ]:
# ============================
# OPTIONAL — Prompt ablation comparison table
# ============================

if "prompt_variant" in df_summary.columns:
    pivot_cols = ["dataset_key", "model_id", "prompt_variant", "auroc_c_prob_avg", "auroc_c_verb", "aurc_c_prob_avg", "aurc_c_verb", "accuracy_avg"]
    df_prompt_compare = df_summary[pivot_cols].copy()
    df_prompt_compare = df_prompt_compare.sort_values(["dataset_key", "model_id", "prompt_variant"]).reset_index(drop=True)
    display(df_prompt_compare)
else:
    print("No prompt_variant column found in df_summary.")


## Paper artifacts

Run the cells below after the full evaluation has finished and `summary_metrics.csv` has been written.
They regenerate the tables and figures used in the paper from saved outputs.


## Output layout


Output files are written under `OUTPUT_DIR`.

Main evaluation outputs:
- `summary_metrics.csv`
- `<dataset>/<model>/prompt_variant=<variant>/results.csv`
- `<dataset>/<model>/prompt_variant=<variant>/metrics.json`

Paper artifacts:
- `paper_artifacts/` with LaTeX tables, CSV exports, and figures.


In [ ]:
# ============================
# SETUP FOR PAPER ARTIFACTS V2
# Run this once first
# ============================

import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use your existing OUTPUT_DIR if already defined; otherwise set it here
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = "/content/self_verify_runs"

ART_DIR = os.path.join(OUTPUT_DIR, "paper_artifacts")
os.makedirs(ART_DIR, exist_ok=True)

MODELS = [
    "microsoft/phi-2",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
]

DATASETS = ["truthfulqa_mc", "ai2_arc_challenge"]
PROMPT_VARIANTS = ["default", "audit_v1"]

def safe_name(s: str) -> str:
    return s.replace("/", "__").replace(":", "_").replace(" ", "_")

def pretty_dataset(x):
    return {
        "truthfulqa_mc": "TruthfulQA-MC",
        "ai2_arc_challenge": "ARC-Challenge",
    }.get(x, x)

def pretty_model(x):
    name = x.split("/")[-1]
    return {
        "phi-2": "Phi-2",
        "Qwen2.5-1.5B-Instruct": "Qwen-1.5B",
        "Qwen2.5-7B-Instruct": "Qwen-7B",
        "TinyLlama-1.1B-Chat-v1.0": "TinyLlama-1.1B",
        "DeepSeek-R1-Distill-Llama-8B": "DeepSeek-R1-Distill-8B",
    }.get(name, name)

# Load summary metrics
summary_path = os.path.join(OUTPUT_DIR, "summary_metrics.csv")
summary_df = pd.read_csv(summary_path)

summary_df["Dataset"] = summary_df["dataset_key"].map(pretty_dataset)
summary_df["Model"] = summary_df["model_id"].map(pretty_model)

# Load all per-example results.csv files (prompt-aware)
rows = []
missing = []

for dkey in DATASETS:
    for mid in MODELS:
        for prompt_variant in PROMPT_VARIANTS:
            p = os.path.join(
                OUTPUT_DIR,
                dkey,
                safe_name(mid),
                f"prompt_variant={safe_name(prompt_variant)}",
                "results.csv",
            )
            if not os.path.exists(p):
                missing.append(p)
                continue
            df = pd.read_csv(p)
            df["dataset_key"] = dkey
            df["model_id"] = mid
            df["prompt_variant"] = prompt_variant
            rows.append(df)

if missing:
    print("[WARNING] Missing files:")
    for m in missing[:10]:
        print(" ", m)
    if len(missing) > 10:
        print(f" ... and {len(missing)-10} more")

all_df = pd.concat(rows, ignore_index=True)
all_df["Dataset"] = all_df["dataset_key"].map(pretty_dataset)
all_df["Model"] = all_df["model_id"].map(pretty_model)

print("Loaded summary_df:", summary_df.shape)
print("Loaded all_df:", all_df.shape)
print("Artifacts will save to:", ART_DIR)

In [ ]:
# ============================
# AUXILIARY BASELINES FOR APPENDIX
# margin / entropy / temperature-scaled LL-AVG
# Expects: all_df, SEED, OUTPUT_DIR or ART_DIR already defined
# ============================

import os
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss

# ----------------------------
# helpers
# ----------------------------
def parse_list(x):
    if isinstance(x, list):
        return np.array(x, dtype=np.float64)
    if isinstance(x, np.ndarray):
        return x.astype(np.float64)
    if pd.isna(x):
        return np.array([], dtype=np.float64)
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=np.float64)
    return np.array(x, dtype=np.float64)

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    denom = np.sum(ex)
    if denom <= 0 or not np.isfinite(denom):
        return np.ones_like(x) / len(x)
    return ex / denom

def auroc_safe(conf, y):
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, conf))

def brier(conf, y):
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return float(np.mean((conf - y) ** 2))

def ece(conf, y, n_bins=10):
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(conf)
    if total == 0:
        return np.nan
    out = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)
        if not np.any(mask):
            continue
        acc = y[mask].mean()
        avg_conf = conf[mask].mean()
        out += (mask.mean()) * abs(acc - avg_conf)
    return float(out)

def aurc(conf, y):
    """
    Area under risk-coverage curve.
    Lower is better.
    """
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    order = np.argsort(-conf)  # descending confidence
    y_sorted = y[order]
    n = len(y_sorted)
    if n == 0:
        return np.nan
    coverages = np.arange(1, n + 1, dtype=np.float64) / n
    risks = 1.0 - np.cumsum(y_sorted) / np.arange(1, n + 1, dtype=np.float64)
    return float(np.trapz(risks, coverages))

def error_at_coverage(conf, y, coverage):
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    n = len(y)
    if n == 0:
        return np.nan
    k = max(1, int(np.ceil(coverage * n)))
    order = np.argsort(-conf)
    keep = order[:k]
    return float(1.0 - y[keep].mean())

def coverage_at_error(conf, y, max_error):
    conf = np.asarray(conf, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    n = len(y)
    if n == 0:
        return np.nan
    order = np.argsort(-conf)
    y_sorted = y[order]
    risks = 1.0 - np.cumsum(y_sorted) / np.arange(1, n + 1, dtype=np.float64)
    coverages = np.arange(1, n + 1, dtype=np.float64) / n
    valid = np.where(risks <= max_error)[0]
    if len(valid) == 0:
        return 0.0
    return float(coverages[valid[-1]])

def confidence_from_margin(p):
    p = np.asarray(p, dtype=np.float64)
    if len(p) == 0:
        return np.nan
    s = np.sort(p)[::-1]
    top1 = s[0]
    top2 = s[1] if len(s) > 1 else 0.0
    return float(top1 - top2)

def confidence_from_entropy(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, 1.0)
    p = p / p.sum()
    k = len(p)
    if k <= 1:
        return 1.0
    ent = -np.sum(p * np.log(p))
    ent_max = np.log(k)
    return float(1.0 - ent / ent_max)

def apply_temperature(scores, T):
    scores = np.asarray(scores, dtype=np.float64)
    T = max(float(T), 1e-4)
    return softmax_np(scores / T)

def fit_temperature(scores_list, gold, calib_frac=0.2, seed=42, min_calib=50, grid=None):
    """
    Fits temperature on a held-out calibration subset by minimizing multiclass NLL.
    scores_list: list of unnormalized per-choice AVG scores
    gold: gold indices
    """
    n = len(scores_list)
    if n == 0:
        return 1.0

    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_calib = max(min_calib, int(np.ceil(calib_frac * n)))
    n_calib = min(n_calib, n)
    calib_idx = idx[:n_calib]

    calib_scores = [np.asarray(scores_list[i], dtype=np.float64) for i in calib_idx]
    calib_gold = np.asarray([gold[i] for i in calib_idx], dtype=np.int64)

    if grid is None:
        # Dense enough for appendix comparisons without adding scipy dependency
        grid = np.concatenate([
            np.linspace(0.25, 1.50, 26),
            np.linspace(1.6, 4.0, 13)
        ])

    best_T = 1.0
    best_nll = np.inf

    for T in grid:
        probs = np.vstack([apply_temperature(s, T) for s in calib_scores])
        try:
            nll = log_loss(calib_gold, probs, labels=np.arange(probs.shape[1]))
        except ValueError:
            continue
        if nll < best_nll:
            best_nll = nll
            best_T = float(T)

    return best_T

def export_latex_table(df, out_path, caption, label, floatfmt="%.3f"):
    cols = list(df.columns)
    colfmt = "ll" + "r" * (len(cols) - 2)
    latex = df.to_latex(
        index=False,
        escape=False,
        float_format=(lambda x: floatfmt % x if isinstance(x, (float, np.floating)) and np.isfinite(x) else str(x)),
        column_format=colfmt,
        caption=caption,
        label=label,
    )
    with open(out_path, "w") as f:
        f.write(latex)

# ----------------------------
# choose output directory
# ----------------------------
if "ART_DIR" in globals():
    aux_dir = ART_DIR
elif "OUTPUT_DIR" in globals():
    aux_dir = os.path.join(OUTPUT_DIR, "paper_artifacts")
else:
    aux_dir = "./paper_artifacts"
os.makedirs(aux_dir, exist_ok=True)

# ----------------------------
# compute baselines
# ----------------------------
required_cols = [
    "dataset_key",
    "model_id",
    "prompt_variant",
    "choice_probs_avg",
    "choice_scores",
    "choice_tok_lens",
    "gold_index",
    "pred_index_avg",
    "c_prob_avg",
    "c_verb",
]
missing = [c for c in required_cols if c not in all_df.columns]
if missing:
    raise ValueError(f"all_df is missing required columns: {missing}")

rows = []

group_cols = ["dataset_key", "model_id", "prompt_variant"]

for (dkey, mid, pvar), d in all_df.groupby(group_cols):
    d = d.reset_index(drop=True).copy()

    probs_avg = d["choice_probs_avg"].apply(parse_list).tolist()
    scores_sum = d["choice_scores"].apply(parse_list).tolist()
    tok_lens = d["choice_tok_lens"].apply(parse_list).tolist()

    # reconstruct LL-AVG option scores from summed token logprobs / token lengths
    scores_avg = [
        scores_sum[i] / np.maximum(tok_lens[i], 1.0)
        for i in range(len(d))
    ]

    gold = d["gold_index"].values.astype(int)
    pred_avg = d["pred_index_avg"].values.astype(int)
    y_avg = (pred_avg == gold).astype(int)

    # Aux baseline 1: top-2 probability margin on LL-AVG probs
    conf_margin = np.array([confidence_from_margin(p) for p in probs_avg], dtype=np.float64)

    # Aux baseline 2: 1 - normalized entropy on LL-AVG probs
    conf_entropy = np.array([confidence_from_entropy(p) for p in probs_avg], dtype=np.float64)

    # Aux baseline 3: temperature-scaled LL-AVG
    T = fit_temperature(scores_avg, gold, calib_frac=0.2, seed=SEED, min_calib=50)
    probs_temp = [apply_temperature(scores_avg[i], T) for i in range(len(d))]
    conf_temp = np.array([p[pred_avg[i]] for i, p in enumerate(probs_temp)], dtype=np.float64)

    # Existing main signals for reference
    conf_llavg = d["c_prob_avg"].values.astype(np.float64)
    conf_sv = d["c_verb"].values.astype(np.float64)

    signal_map = {
        "LL-AVG": conf_llavg,
        "Self-Verify": conf_sv,
        "Margin": conf_margin,
        "EntropyConf": conf_entropy,
        "LL-AVG-T": conf_temp,
    }

    for signal_name, conf in signal_map.items():
        rows.append({
            "dataset_key": dkey,
            "model_id": mid,
            "prompt_variant": pvar,
            "signal": signal_name,
            "n": int(len(d)),
            "T_LLAVG": float(T) if signal_name == "LL-AVG-T" else np.nan,
            "AUROC": auroc_safe(conf, y_avg),
            "AURC": aurc(conf, y_avg),
            "Brier": brier(conf, y_avg),
            "ECE10": ece(conf, y_avg, 10),
            "err@80cov": error_at_coverage(conf, y_avg, 0.80),
            "err@50cov": error_at_coverage(conf, y_avg, 0.50),
            "cov@<=20err": coverage_at_error(conf, y_avg, 0.20),
            "cov@<=10err": coverage_at_error(conf, y_avg, 0.10),
        })

aux_baselines_long = pd.DataFrame(rows)

# ----------------------------
# wide appendix table
# ----------------------------
metric_order = ["AUROC", "AURC", "Brier", "ECE10", "err@80cov", "err@50cov", "cov@<=20err", "cov@<=10err"]

aux_baselines_wide = (
    aux_baselines_long
    .pivot_table(
        index=["dataset_key", "model_id", "prompt_variant"],
        columns="signal",
        values=metric_order,
        aggfunc="first"
    )
)

# flatten multiindex cols
aux_baselines_wide.columns = [
    f"{metric}_{signal}".replace("Self-Verify", "SelfVerify").replace("EntropyConf", "Entropy")
    for metric, signal in aux_baselines_wide.columns
]
aux_baselines_wide = aux_baselines_wide.reset_index()

# optional: compact summary table for appendix
compact_cols = [
    "dataset_key", "model_id", "prompt_variant",
    "AUROC_LL-AVG", "AUROC_SelfVerify", "AUROC_Margin", "AUROC_Entropy", "AUROC_LL-AVG-T",
    "AURC_LL-AVG", "AURC_SelfVerify", "AURC_Margin", "AURC_Entropy", "AURC_LL-AVG-T",
]
compact_cols = [c for c in compact_cols if c in aux_baselines_wide.columns]
aux_baselines_compact = aux_baselines_wide[compact_cols].copy()

# ----------------------------
# save artifacts
# ----------------------------
csv_long = os.path.join(aux_dir, "appendix_aux_baselines_long.csv")
csv_wide = os.path.join(aux_dir, "appendix_aux_baselines_wide.csv")
csv_compact = os.path.join(aux_dir, "appendix_aux_baselines_compact.csv")
tex_compact = os.path.join(aux_dir, "appendix_table_aux_baselines.tex")

aux_baselines_long.to_csv(csv_long, index=False)
aux_baselines_wide.to_csv(csv_wide, index=False)
aux_baselines_compact.to_csv(csv_compact, index=False)

export_latex_table(
    aux_baselines_compact,
    tex_compact,
    caption="Auxiliary confidence baselines computed from the multiple-choice answer distribution. Margin denotes the gap between the top two LL-AVG option probabilities; Entropy denotes one minus normalized predictive entropy; LL-AVG-T denotes temperature-scaled LL-AVG fit on a held-out 20\\% calibration split.",
    label="tab:appendix_aux_baselines",
    floatfmt="%.3f",
)

print("Saved:")
print(" -", csv_long)
print(" -", csv_wide)
print(" -", csv_compact)
print(" -", tex_compact)

aux_baselines_compact.sort_values(["dataset_key", "prompt_variant", "model_id"]).reset_index(drop=True)

In [ ]:
# ============================
# TABLE 1 — MAIN METRICS TABLE
# ============================

table1 = summary_df.copy()

table1 = table1[[
    "Dataset", "Model", "prompt_variant",
    "accuracy_avg",
    "auroc_c_prob_avg", "auroc_c_verb", "auroc_c_prob_sum",
    "aurc_c_prob_avg", "aurc_c_verb", "aurc_c_prob_sum",
]].rename(columns={
    "prompt_variant": "Prompt",
    "accuracy_avg": "Acc (LL-AVG)",
    "auroc_c_prob_avg": "AUROC (LL-AVG)",
    "auroc_c_verb": "AUROC (Self-Verify)",
    "auroc_c_prob_sum": "AUROC (LL-SUM)",
    "aurc_c_prob_avg": "AURC (LL-AVG)",
    "aurc_c_verb": "AURC (Self-Verify)",
    "aurc_c_prob_sum": "AURC (LL-SUM)",
})

for c in table1.columns:
    if c not in ["Dataset", "Model", "Prompt"]:
        table1[c] = table1[c].astype(float).round(3)

table1 = table1.sort_values(["Dataset", "Model", "Prompt"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "table1_main_metrics.csv")
tex_path = os.path.join(ART_DIR, "table1_main_metrics.tex")

table1.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(table1.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
table1

In [ ]:
# ============================
# TABLE 2 — DELTA TABLE
# ============================

table2 = summary_df.copy()

table2["delta_auroc_sv_vs_avg"] = table2["auroc_c_verb"] - table2["auroc_c_prob_avg"]
table2["delta_aurc_sv_vs_avg"] = table2["aurc_c_verb"] - table2["aurc_c_prob_avg"]
table2["delta_auroc_sv_vs_sum"] = table2["auroc_c_verb"] - table2["auroc_c_prob_sum"]
table2["delta_aurc_sv_vs_sum"] = table2["aurc_c_verb"] - table2["aurc_c_prob_sum"]

table2 = table2[[
    "Dataset", "Model", "prompt_variant",
    "delta_auroc_sv_vs_avg",
    "delta_aurc_sv_vs_avg",
    "delta_auroc_sv_vs_sum",
    "delta_aurc_sv_vs_sum",
]].rename(columns={
    "prompt_variant": "Prompt",
    "delta_auroc_sv_vs_avg": "ΔAUROC (SV-LLAVG)",
    "delta_aurc_sv_vs_avg": "ΔAURC (SV-LLAVG)",
    "delta_auroc_sv_vs_sum": "ΔAUROC (SV-LLSUM)",
    "delta_aurc_sv_vs_sum": "ΔAURC (SV-LLSUM)",
})

for c in table2.columns:
    if c not in ["Dataset", "Model", "Prompt"]:
        table2[c] = table2[c].astype(float).round(3)

table2 = table2.sort_values(["Dataset", "Model", "Prompt"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "table2_deltas.csv")
tex_path = os.path.join(ART_DIR, "table2_deltas.tex")

table2.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(table2.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
table2

In [ ]:
# ============================
# TABLE 3 — BOOTSTRAP CI TABLE
# ΔAUROC(Self-Verify - LL-AVG)
# ============================

from sklearn.metrics import roc_auc_score

def bootstrap_delta_auroc(y, conf_sv, conf_avg, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=int)
    conf_sv = np.asarray(conf_sv, dtype=float)
    conf_avg = np.asarray(conf_avg, dtype=float)

    deltas = []
    n = len(y)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        y_b = y[idx]
        if len(np.unique(y_b)) < 2:
            continue
        sv_b = conf_sv[idx]
        avg_b = conf_avg[idx]
        delta = roc_auc_score(y_b, sv_b) - roc_auc_score(y_b, avg_b)
        deltas.append(delta)

    deltas = np.asarray(deltas, dtype=float)
    return {
        "delta_mean": float(deltas.mean()) if len(deltas) else np.nan,
        "ci_lower": float(np.percentile(deltas, 2.5)) if len(deltas) else np.nan,
        "ci_upper": float(np.percentile(deltas, 97.5)) if len(deltas) else np.nan,
        "n_boot_valid": int(len(deltas)),
    }

rows = []
for (dkey, mid, prompt), d in all_df.groupby(["dataset_key", "model_id", "prompt_variant"]):
    y = d["y_avg"].values.astype(int)
    conf_sv = d["c_verb"].values.astype(float)
    conf_avg = d["c_prob_avg"].values.astype(float)

    out = bootstrap_delta_auroc(y, conf_sv, conf_avg, n_boot=2000, seed=42)
    rows.append({
        "Dataset": pretty_dataset(dkey),
        "Model": pretty_model(mid),
        "Prompt": prompt,
        "ΔAUROC (SV-LLAVG)": round(out["delta_mean"], 3),
        "CI lower": round(out["ci_lower"], 3),
        "CI upper": round(out["ci_upper"], 3),
        "n_boot_valid": out["n_boot_valid"],
    })

table3 = pd.DataFrame(rows).sort_values(["Dataset", "Model", "Prompt"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "table3_bootstrap_delta_auroc.csv")
tex_path = os.path.join(ART_DIR, "table3_bootstrap_delta_auroc.tex")

table3.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(table3.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
table3

In [ ]:
# ============================
# TABLE 4 — SELECTIVE PREDICTION OPERATING POINTS
# Includes LL-AVG, Self-Verify, LL-SUM
# ============================

def error_at_coverage(conf, y, coverage):
    conf = np.asarray(conf, float)
    y = np.asarray(y, int)
    n = len(y)
    k = max(1, int(np.floor(coverage * n)))
    idx = np.argsort(-conf)[:k]
    return float(1.0 - y[idx].mean())

def coverage_at_error(conf, y, target_err):
    conf = np.asarray(conf, float)
    y = np.asarray(y, int)
    idx = np.argsort(-conf)
    y_s = y[idx]
    risk = 1.0 - (np.cumsum(y_s) / np.arange(1, len(y_s) + 1))
    ok = np.where(risk <= target_err)[0]
    if len(ok) == 0:
        return 0.0
    return float((ok[-1] + 1) / len(y_s))

ops_rows = []
for (dkey, mid, prompt), d in all_df.groupby(["dataset_key", "model_id", "prompt_variant"]):
    # LL-AVG and SV are evaluated against y_avg
    y_avg = d["y_avg"].values.astype(int)
    # LL-SUM against y_sum
    y_sum = d["y_sum"].values.astype(int)

    signal_specs = [
        ("LL-AVG", "c_prob_avg", y_avg),
        ("Self-Verify", "c_verb", y_avg),
        ("LL-SUM", "c_prob_sum", y_sum),
    ]

    for signal_name, conf_col, y in signal_specs:
        conf = d[conf_col].values.astype(float)
        ops_rows.append({
            "Dataset": pretty_dataset(dkey),
            "Model": pretty_model(mid),
            "Prompt": prompt,
            "Signal": signal_name,
            "err@80%cov": round(error_at_coverage(conf, y, 0.8), 3),
            "err@50%cov": round(error_at_coverage(conf, y, 0.5), 3),
            "cov@20%err": round(coverage_at_error(conf, y, 0.2), 3),
            "cov@10%err": round(coverage_at_error(conf, y, 0.1), 3),
        })

table4 = pd.DataFrame(ops_rows).sort_values(["Dataset", "Model", "Prompt", "Signal"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "table4_operating_points.csv")
tex_path = os.path.join(ART_DIR, "table4_operating_points.tex")

table4.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(table4.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
table4

In [ ]:
# ============================
# TABLE 5 — PROMPT ABLATION SUMMARY
# Self-Verify only
# ============================

sv = summary_df.copy()

table5 = sv.pivot_table(
    index=["Dataset", "Model"],
    columns="prompt_variant",
    values=["auroc_c_verb", "aurc_c_verb"],
).reset_index()

# Flatten columns
table5.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in table5.columns
]

# Rename
rename_map = {
    "auroc_c_verb_default": "AUROC default",
    "auroc_c_verb_audit_v1": "AUROC audit_v1",
    "aurc_c_verb_default": "AURC default",
    "aurc_c_verb_audit_v1": "AURC audit_v1",
}
table5 = table5.rename(columns=rename_map)

table5["|ΔAUROC|"] = (table5["AUROC default"] - table5["AUROC audit_v1"]).abs()
table5["|ΔAURC|"] = (table5["AURC default"] - table5["AURC audit_v1"]).abs()

for c in table5.columns:
    if c not in ["Dataset", "Model"]:
        table5[c] = table5[c].astype(float).round(3)

table5 = table5.sort_values(["Dataset", "Model"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "table5_prompt_ablation.csv")
tex_path = os.path.join(ART_DIR, "table5_prompt_ablation.tex")

table5.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(table5.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
table5

In [ ]:
# ============================
# FIGURE 1 — AUROC COMPARISON
# Separate file per dataset × prompt
# ============================

plot_df = summary_df.copy()

for dataset_key in DATASETS:
    for prompt in PROMPT_VARIANTS:
        d = plot_df[
            (plot_df["dataset_key"] == dataset_key) &
            (plot_df["prompt_variant"] == prompt)
        ].copy()

        d = d.sort_values("Model")
        x = np.arange(len(d))
        width = 0.25

        plt.figure(figsize=(9, 4.5))
        plt.bar(x - width, d["auroc_c_prob_avg"].values, width, label="LL-AVG")
        plt.bar(x,         d["auroc_c_verb"].values,     width, label="Self-Verify")
        plt.bar(x + width, d["auroc_c_prob_sum"].values, width, label="LL-SUM")

        plt.xticks(x, d["Model"].values, rotation=20)
        plt.ylim(0, 1)
        plt.ylabel("AUROC")
        plt.title(f"AUROC by confidence signal — {pretty_dataset(dataset_key)} ({prompt})")
        plt.legend()
        plt.tight_layout()

        out = os.path.join(ART_DIR, f"figure1_auroc_{dataset_key}_{prompt}.png")
        plt.savefig(out, dpi=250, bbox_inches="tight")
        plt.close()
        print("Saved:", out)

In [ ]:
# ============================
# FIGURE 2 — AURC COMPARISON
# Separate file per dataset × prompt
# ============================

plot_df = summary_df.copy()

for dataset_key in DATASETS:
    for prompt in PROMPT_VARIANTS:
        d = plot_df[
            (plot_df["dataset_key"] == dataset_key) &
            (plot_df["prompt_variant"] == prompt)
        ].copy()

        d = d.sort_values("Model")
        x = np.arange(len(d))
        width = 0.25

        plt.figure(figsize=(9, 4.5))
        plt.bar(x - width, d["aurc_c_prob_avg"].values, width, label="LL-AVG")
        plt.bar(x,         d["aurc_c_verb"].values,     width, label="Self-Verify")
        plt.bar(x + width, d["aurc_c_prob_sum"].values, width, label="LL-SUM")

        plt.xticks(x, d["Model"].values, rotation=20)
        plt.ylabel("AURC (lower is better)")
        plt.title(f"AURC by confidence signal — {pretty_dataset(dataset_key)} ({prompt})")
        plt.legend()
        plt.tight_layout()

        out = os.path.join(ART_DIR, f"figure2_aurc_{dataset_key}_{prompt}.png")
        plt.savefig(out, dpi=250, bbox_inches="tight")
        plt.close()
        print("Saved:", out)

In [ ]:
# ============================
# FIGURE 3 — REPRESENTATIVE RISK-COVERAGE CURVES
# 4 panels:
#   ARC / Qwen-7B
#   TruthfulQA / Qwen-7B
#   ARC / DeepSeek-R1-Distill-8B
#   TruthfulQA / DeepSeek-R1-Distill-8B
# Uses default prompt for Self-Verify
# ============================

def risk_coverage_curve(conf, y):
    conf = np.asarray(conf, float)
    y = np.asarray(y, int)

    idx = np.argsort(-conf)
    y_sorted = y[idx]
    coverage = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
    risk = 1.0 - (np.cumsum(y_sorted) / np.arange(1, len(y_sorted) + 1))
    return coverage, risk

cases = [
    ("ai2_arc_challenge", "Qwen/Qwen2.5-7B-Instruct"),
    ("truthfulqa_mc", "Qwen/Qwen2.5-7B-Instruct"),
    ("ai2_arc_challenge", "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"),
    ("truthfulqa_mc", "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (dkey, mid) in zip(axes, cases):
    d = all_df[
        (all_df["dataset_key"] == dkey) &
        (all_df["model_id"] == mid) &
        (all_df["prompt_variant"] == "default")
    ].copy()

    # LL-AVG and SV use y_avg
    cov, risk = risk_coverage_curve(d["c_prob_avg"].values, d["y_avg"].values)
    ax.plot(cov, risk, label="LL-AVG")

    cov, risk = risk_coverage_curve(d["c_verb"].values, d["y_avg"].values)
    ax.plot(cov, risk, label="Self-Verify")

    # LL-SUM uses y_sum
    cov, risk = risk_coverage_curve(d["c_prob_sum"].values, d["y_sum"].values)
    ax.plot(cov, risk, label="LL-SUM")

    ax.set_title(f"{pretty_dataset(dkey)} | {pretty_model(mid)}")
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Risk")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
fig.tight_layout(rect=[0, 0, 1, 0.95])

out = os.path.join(ART_DIR, "figure3_risk_coverage_representative.png")
plt.savefig(out, dpi=250, bbox_inches="tight")
plt.close()
print("Saved:", out)

In [ ]:
# ============================
# FIGURE 4 — PROMPT ABLATION PLOT
# Self-Verify AUROC by prompt
# One 2-panel figure: ARC and TruthfulQA
# ============================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, dataset_key in zip(axes, DATASETS):
    d = summary_df[summary_df["dataset_key"] == dataset_key].copy()

    models_order = [pretty_model(m) for m in MODELS]
    d["Model"] = pd.Categorical(d["Model"], categories=models_order, ordered=True)
    d = d.sort_values(["Model", "prompt_variant"])

    x = np.arange(len(models_order))
    width = 0.35

    d_default = d[d["prompt_variant"] == "default"].set_index("Model").reindex(models_order)
    d_audit   = d[d["prompt_variant"] == "audit_v1"].set_index("Model").reindex(models_order)

    ax.bar(x - width/2, d_default["auroc_c_verb"].values, width, label="default")
    ax.bar(x + width/2, d_audit["auroc_c_verb"].values,   width, label="audit_v1")

    ax.set_xticks(x)
    ax.set_xticklabels(models_order, rotation=20)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Self-Verify AUROC")
    ax.set_title(pretty_dataset(dataset_key))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.tight_layout(rect=[0, 0, 1, 0.92])

out = os.path.join(ART_DIR, "figure4_prompt_ablation_auroc.png")
plt.savefig(out, dpi=250, bbox_inches="tight")
plt.close()
print("Saved:", out)

In [ ]:
# ============================
# FIGURE 5 — SCALE / FAMILY EFFECT PLOT
# Shows LL-AVG vs Self-Verify AUROC for selected models
# default prompt only
# ============================

selected_models = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, dataset_key in zip(axes, DATASETS):
    d = summary_df[
        (summary_df["dataset_key"] == dataset_key) &
        (summary_df["prompt_variant"] == "default") &
        (summary_df["model_id"].isin(selected_models))
    ].copy()

    d = d.sort_values("model_id")
    labels = [pretty_model(m) for m in d["model_id"]]
    x = np.arange(len(d))

    ax.plot(x, d["auroc_c_prob_avg"].values, marker="o", label="LL-AVG")
    ax.plot(x, d["auroc_c_verb"].values, marker="o", label="Self-Verify")
    ax.plot(x, d["auroc_c_prob_sum"].values, marker="o", label="LL-SUM")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20)
    ax.set_ylim(0, 1)
    ax.set_ylabel("AUROC")
    ax.set_title(pretty_dataset(dataset_key))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
fig.tight_layout(rect=[0, 0, 1, 0.92])

out = os.path.join(ART_DIR, "figure5_scale_family_effects.png")
plt.savefig(out, dpi=250, bbox_inches="tight")
plt.close()
print("Saved:", out)

In [ ]:
# ============================
# OPTIONAL APPENDIX TABLE — CALIBRATION
# ============================

cal = summary_df.copy()

cal = cal[[
    "Dataset", "Model", "prompt_variant",
    "brier_c_prob_avg", "brier_c_verb", "brier_c_prob_sum",
    "ece10_c_prob_avg", "ece10_c_verb", "ece10_c_prob_sum",
]].rename(columns={
    "prompt_variant": "Prompt",
    "brier_c_prob_avg": "Brier (LL-AVG)",
    "brier_c_verb": "Brier (Self-Verify)",
    "brier_c_prob_sum": "Brier (LL-SUM)",
    "ece10_c_prob_avg": "ECE10 (LL-AVG)",
    "ece10_c_verb": "ECE10 (Self-Verify)",
    "ece10_c_prob_sum": "ECE10 (LL-SUM)",
})

for c in cal.columns:
    if c not in ["Dataset", "Model", "Prompt"]:
        cal[c] = cal[c].astype(float).round(3)

cal = cal.sort_values(["Dataset", "Model", "Prompt"]).reset_index(drop=True)

csv_path = os.path.join(ART_DIR, "appendix_table_calibration.csv")
tex_path = os.path.join(ART_DIR, "appendix_table_calibration.tex")

cal.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(cal.to_latex(index=False))

print("Saved:", csv_path)
print("Saved:", tex_path)
cal